In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

PyTorch: 2.10.0+cu128
CUDA: True
GPU count: 2
0 Tesla T4
1 Tesla T4


In [3]:
!nvidia-smi

Sat Sep 12 03:26:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import os

In [5]:
import pandas as pd

BASE = "/kaggle/input/datasets/gillesokhin/nutrition5k-dataset"

nutrition = pd.read_csv(f"{BASE}/dish_nutrition_values.csv")

print(nutrition.columns.tolist())
display(nutrition.head())

['dish_id', 'calories', 'mass', 'fat', 'carb', 'protein']


,dish_id,calories,mass,fat,carb,protein
0,dish_1561662216,300.794281,193.0,12.387489,28.218290,18.633970
1,dish_1562688426,137.569992,88.0,8.256000,5.190000,10.297000
2,dish_1561662054,419.438782,292.0,23.838249,26.351543,25.910593
3,dish_1562008979,382.936646,290.0,22.224644,10.173570,35.345387
4,dish_1560455030,20.590000,103.0,0.148000,4.625000,0.956000


In [6]:
import os

nutrition["image_path"] = nutrition["dish_id"].apply(
    lambda x: f"{BASE}/imagery/realsense_overhead/{x}/rgb.png"
)

# Chỉ giữ món thực sự có ảnh RGB
nutrition = nutrition[
    nutrition["image_path"].apply(os.path.exists)
].reset_index(drop=True)

print("Số món có ảnh:", len(nutrition))

display(
    nutrition[
        ["dish_id", "image_path", "calories"]
    ].head()
)

Số món có ảnh: 3244


,dish_id,image_path,calories
0,dish_1561662216,/kaggle/input/datasets/gillesokhin/nutrition5k...,300.794281
1,dish_1561662054,/kaggle/input/datasets/gillesokhin/nutrition5k...,419.438782
2,dish_1562008979,/kaggle/input/datasets/gillesokhin/nutrition5k...,382.936646
3,dish_1560455030,/kaggle/input/datasets/gillesokhin/nutrition5k...,20.590000
4,dish_1558372433,/kaggle/input/datasets/gillesokhin/nutrition5k...,74.360001


In [7]:
print(nutrition.columns.tolist())
display(nutrition.head())

['dish_id', 'calories', 'mass', 'fat', 'carb', 'protein', 'image_path']


,dish_id,calories,mass,fat,carb,protein,image_path
0,dish_1561662216,300.794281,193.0,12.387489,28.218290,18.633970,/kaggle/input/datasets/gillesokhin/nutrition5k...
1,dish_1561662054,419.438782,292.0,23.838249,26.351543,25.910593,/kaggle/input/datasets/gillesokhin/nutrition5k...
2,dish_1562008979,382.936646,290.0,22.224644,10.173570,35.345387,/kaggle/input/datasets/gillesokhin/nutrition5k...
3,dish_1560455030,20.590000,103.0,0.148000,4.625000,0.956000,/kaggle/input/datasets/gillesokhin/nutrition5k...
4,dish_1558372433,74.360001,143.0,0.286000,0.429000,20.020000,/kaggle/input/datasets/gillesokhin/nutrition5k...


In [8]:
df = nutrition[
    ["dish_id", "image_path", "calories"]
].copy()

display(df.head())
print("Tổng số mẫu:", len(df))

,dish_id,image_path,calories
0,dish_1561662216,/kaggle/input/datasets/gillesokhin/nutrition5k...,300.794281
1,dish_1561662054,/kaggle/input/datasets/gillesokhin/nutrition5k...,419.438782
2,dish_1562008979,/kaggle/input/datasets/gillesokhin/nutrition5k...,382.936646
3,dish_1560455030,/kaggle/input/datasets/gillesokhin/nutrition5k...,20.590000
4,dish_1558372433,/kaggle/input/datasets/gillesokhin/nutrition5k...,74.360001


Tổng số mẫu: 3244


In [9]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 2270
Val: 487
Test: 487


In [10]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

class NutritionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        calories = torch.tensor(row["calories"], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, calories

In [11]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [12]:
train_dataset = NutritionDataset(train_df, train_transform)
val_dataset = NutritionDataset(val_df, val_transform)
test_dataset = NutritionDataset(test_df, val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(len(train_dataset), len(val_dataset), len(test_dataset))

2270 487 487


In [13]:
images, calories = next(iter(train_loader))

print("Images:", images.shape)
print("Calories:", calories.shape)
print("Ví dụ calories:", calories[:5])

Images: torch.Size([64, 3, 224, 224])
Calories: torch.Size([64])
Ví dụ calories: tensor([482.9831, 352.5292,  63.7200, 181.4897, 414.7332])


In [14]:
# Lưu split cố định để mọi experiment sau dùng cùng dữ liệu
train_df.to_csv("/kaggle/working/train.csv", index=False)
val_df.to_csv("/kaggle/working/val.csv", index=False)
test_df.to_csv("/kaggle/working/test.csv", index=False)

print("Saved fixed splits:")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))


Saved fixed splits:
Train: 2270
Validation: 487
Test: 487


In [15]:
# Thống kê mô tả target calories
def calorie_stats(dataframe, split_name):
    return {
        "split": split_name,
        "n_samples": len(dataframe),
        "min_kcal": dataframe["calories"].min(),
        "max_kcal": dataframe["calories"].max(),
        "mean_kcal": dataframe["calories"].mean(),
        "std_kcal": dataframe["calories"].std(),
        "median_kcal": dataframe["calories"].median(),
    }

stats_df = pd.DataFrame([
    calorie_stats(train_df, "Train"),
    calorie_stats(val_df, "Validation"),
    calorie_stats(test_df, "Test"),
    calorie_stats(pd.concat([train_df, val_df, test_df]), "Total"),
])

display(stats_df.round(2))
stats_df.to_csv("/kaggle/working/dataset_calorie_statistics.csv", index=False)


,split,n_samples,min_kcal,max_kcal,mean_kcal,std_kcal,median_kcal
0,Train,2270,0.00,3943.33,253.31,221.40,207.40
1,Validation,487,2.30,1324.08,256.67,212.09,212.16
2,Test,487,1.15,1264.55,266.35,218.93,234.58
3,Total,3244,0.00,3943.33,255.77,219.64,210.32


# EXPERIMENTS

Phần chuẩn bị dữ liệu phía trên được giữ cố định.  
Thêm các experiment mới từ đây trở xuống để bảo đảm tất cả thí nghiệm dùng cùng dataset split.


In [18]:
import pandas as pd
import os

ingredient_path = f"{BASE}/ingredients_metadata.csv"

print("Exists:", os.path.exists(ingredient_path))

ingredients = pd.read_csv(ingredient_path)

print("\nShape:", ingredients.shape)
print("\nColumns:")
print(ingredients.columns.tolist())

display(ingredients.head(10))

Exists: True

Shape: (555, 6)

Columns:
['ingr_name', 'ingr_id', 'cal/g', 'fat(g)', 'carb(g)', 'protein(g)']


,ingr_name,ingr_id,cal/g,fat(g),carb(g),protein(g)
0,cottage cheese,ingr_0000000001,0.980,0.043,0.034,0.110
1,strawberries,ingr_0000000002,0.330,0.003,0.080,0.007
2,garden salad,ingr_0000000003,0.646,0.034,0.032,0.061
3,bacon,ingr_0000000004,5.410,0.420,0.014,0.370
4,potatoes,ingr_0000000005,0.770,0.001,0.170,0.020
5,caesar salad,ingr_0000000006,0.440,0.021,0.043,0.032
6,cauliflower,ingr_0000000007,0.250,0.003,0.050,0.018
7,scrambled eggs,ingr_0000000008,1.480,0.110,0.016,0.100
8,wild rice,ingr_0000000009,1.190,0.004,0.250,0.050
9,steak,ingr_0000000010,2.710,0.190,0.000,0.250


In [19]:
import os

print(os.listdir(BASE))

['dish_nutrition_values.csv', 'ingredients_metadata.csv', 'imagery', 'dish_ingredients.csv']


In [20]:
import glob

files = glob.glob(f"{BASE}/*.csv")

for f in files:
    print(f)

/kaggle/input/datasets/gillesokhin/nutrition5k-dataset/dish_nutrition_values.csv
/kaggle/input/datasets/gillesokhin/nutrition5k-dataset/ingredients_metadata.csv
/kaggle/input/datasets/gillesokhin/nutrition5k-dataset/dish_ingredients.csv


In [21]:
dish_ingredients_path = f"{BASE}/dish_ingredients.csv"

dish_ingredients = pd.read_csv(
    dish_ingredients_path
)

print("Shape:", dish_ingredients.shape)

print("\nColumns:")
print(dish_ingredients.columns.tolist())

display(
    dish_ingredients.head(15)
)

Shape: (27225, 8)

Columns:
['dish_id', 'ingr_id', 'ingr_name', 'grams', 'calories', 'fat', 'carb', 'protein']


,dish_id,ingr_id,ingr_name,grams,calories,fat,carb,protein
0,dish_1561662216,ingr_0000000508,soy sauce,3.398568,1.801241,0.020391,0.166530,0.275284
1,dish_1561662216,ingr_0000000122,garlic,2.124105,3.164916,0.010621,0.700955,0.135943
2,dish_1561662216,ingr_0000000026,white rice,8.496420,11.045346,0.025489,2.378998,0.229403
3,dish_1561662216,ingr_0000000524,parsley,0.213397,0.079170,0.001707,0.013657,0.006189
4,dish_1561662216,ingr_0000000094,onions,1.707173,0.682869,0.001707,0.153646,0.018779
5,dish_1561662216,ingr_0000000023,brown rice,68.000000,75.480000,0.612000,15.640000,1.768000
6,dish_1561662216,ingr_0000000189,vinegar,0.853586,0.000000,0.000000,0.000000,0.000000
7,dish_1561662216,ingr_0000000054,apple,4.267931,2.219324,0.008536,0.597510,0.012804
8,dish_1561662216,ingr_0000000029,mixed greens,21.339657,5.975104,0.085359,1.131002,0.469472
9,dish_1561662216,ingr_0000000328,sugar,6.372315,24.724582,0.000000,6.372315,0.000000


In [22]:
print(
    dish_ingredients["dish_id"]
    .value_counts()
    .head(10)
)

dish_id
dish_1561578461    34
dish_1561579933    34
dish_1563395510    32
dish_1561060703    32
dish_1561576794    32
dish_1563565508    31
dish_1561574848    30
dish_1563479655    30
dish_1563476577    30
dish_1561580431    29
Name: count, dtype: int64


In [23]:
# ============================================================
# EXP 5 - Step 1
# Build ingredient vocabulary from TRAIN only
# ============================================================

import numpy as np
import pandas as pd

# Chỉ giữ thông tin an toàn
safe_ingredients = (
    dish_ingredients[
        ["dish_id", "ingr_id"]
    ]
    .drop_duplicates()
    .copy()
)

# Các dish thuộc TRAIN
train_dish_ids = set(train_df["dish_id"])

# Ingredient xuất hiện trong TRAIN
train_ingr_ids = (
    safe_ingredients[
        safe_ingredients["dish_id"].isin(train_dish_ids)
    ]["ingr_id"]
    .unique()
)

# 0 = PAD
# 1 = UNK
ingredient_to_idx = {
    ingr_id: idx + 2
    for idx, ingr_id in enumerate(sorted(train_ingr_ids))
}

PAD_IDX = 0
UNK_IDX = 1

VOCAB_SIZE = len(ingredient_to_idx) + 2

print("Ingredient vocabulary size:", VOCAB_SIZE)
print("PAD_IDX:", PAD_IDX)
print("UNK_IDX:", UNK_IDX)

Ingredient vocabulary size: 197
PAD_IDX: 0
UNK_IDX: 1


In [24]:
# ============================================================
# EXP 5 - Step 2
# dish_id -> list of ingredient ids
# ============================================================

dish_to_ingredients = (
    safe_ingredients
    .groupby("dish_id")["ingr_id"]
    .apply(list)
    .to_dict()
)


def add_ingredient_list(df):
    result = df.copy()

    result["ingredient_ids"] = (
        result["dish_id"]
        .map(dish_to_ingredients)
    )

    return result


train_ing_df = add_ingredient_list(train_df)
val_ing_df   = add_ingredient_list(val_df)
test_ing_df  = add_ingredient_list(test_df)

In [25]:
for name, df_ in [
    ("Train", train_ing_df),
    ("Validation", val_ing_df),
    ("Test", test_ing_df)
]:
    has_ing = df_["ingredient_ids"].notna()

    print(
        f"{name}: "
        f"{has_ing.sum()}/{len(df_)} "
        f"({has_ing.mean()*100:.2f}%)"
    )

Train: 2270/2270 (100.00%)
Validation: 487/487 (100.00%)
Test: 487/487 (100.00%)


In [26]:
train_counts = (
    train_ing_df["ingredient_ids"]
    .dropna()
    .apply(len)
)

print("\n=== INGREDIENT COUNT / DISH ===")
print("Min   :", train_counts.min())
print("Mean  :", train_counts.mean())
print("Median:", train_counts.median())
print("Max   :", train_counts.max())

print(
    "Percentiles:",
    np.percentile(
        train_counts,
        [50, 75, 90, 95, 99]
    )
)


=== INGREDIENT COUNT / DISH ===
Min   : 1
Mean  : 7.257709251101321
Median: 4.0
Max   : 34
Percentiles: [ 4. 12. 18. 21. 26.]


In [27]:
MAX_INGREDIENTS = 34

def encode_ingredients(ingredient_list):
    indices = [
        ingredient_to_idx.get(x, UNK_IDX)
        for x in ingredient_list
    ]

    indices = indices[:MAX_INGREDIENTS]

    indices += [PAD_IDX] * (
        MAX_INGREDIENTS - len(indices)
    )

    return indices


for df_ in [
    train_ing_df,
    val_ing_df,
    test_ing_df
]:
    df_["ingredient_indices"] = (
        df_["ingredient_ids"]
        .apply(encode_ingredients)
    )

print(
    train_ing_df[
        ["dish_id", "ingredient_ids", "ingredient_indices"]
    ].head()
)

              dish_id                                     ingredient_ids  \
1898  dish_1565194781  [ingr_0000000437, ingr_0000000291, ingr_000000...   
1188  dish_1559844366  [ingr_0000000077, ingr_0000000027, ingr_000000...   
109   dish_1566849327  [ingr_0000000148, ingr_0000000029, ingr_000000...   
2403  dish_1565020822  [ingr_0000000161, ingr_0000000128, ingr_000000...   
781   dish_1564775147  [ingr_0000000332, ingr_0000000515, ingr_000000...   

                                     ingredient_indices  
1898  [143, 108, 112, 82, 41, 71, 0, 0, 0, 0, 0, 0, ...  
1188  [55, 24, 144, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...  
109   [77, 26, 39, 13, 25, 0, 0, 0, 0, 0, 0, 0, 0, 0...  
2403  [82, 71, 143, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...  
781   [117, 169, 66, 178, 52, 73, 82, 68, 112, 153, ...  


In [28]:
def count_unk(df):
    return sum(
        ids.count(UNK_IDX)
        for ids in df["ingredient_indices"]
    )

print("Train UNK:", count_unk(train_ing_df))
print("Val UNK  :", count_unk(val_ing_df))
print("Test UNK :", count_unk(test_ing_df))

Train UNK: 0
Val UNK  : 6
Test UNK : 3


In [30]:
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class NutritionIngredientDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        if self.transform:
            image = self.transform(image)

        ingredient_ids = torch.tensor(
            row["ingredient_indices"],
            dtype=torch.long
        )

        calories = torch.tensor(
            row["calories"],
            dtype=torch.float32
        )

        return image, ingredient_ids, calories

In [31]:
train_ing_dataset = NutritionIngredientDataset(
    train_ing_df,
    train_transform
)

val_ing_dataset = NutritionIngredientDataset(
    val_ing_df,
    val_transform
)

test_ing_dataset = NutritionIngredientDataset(
    test_ing_df,
    val_transform
)

train_ing_loader = DataLoader(
    train_ing_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_ing_loader = DataLoader(
    val_ing_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [32]:
images, ingredients_batch, calories = next(
    iter(train_ing_loader)
)

print("Images      :", images.shape)
print("Ingredients :", ingredients_batch.shape)
print("Calories    :", calories.shape)

Images      : torch.Size([64, 3, 224, 224])
Ingredients : torch.Size([64, 34])
Calories    : torch.Size([64])


In [33]:
import torch.nn as nn
from torchvision.models import (
    resnet50,
    ResNet50_Weights
)


class VisualIngredientFusion(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=64
    ):
        super().__init__()

        # Visual branch
        self.visual_encoder = resnet50(
            weights=ResNet50_Weights.DEFAULT
        )

        visual_dim = (
            self.visual_encoder.fc.in_features
        )

        self.visual_encoder.fc = nn.Identity()

        # Ingredient branch
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_IDX
        )

        # Fusion
        self.fusion = nn.Sequential(
            nn.Linear(
                visual_dim + embedding_dim,
                512
            ),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1)
        )

    def forward(
        self,
        image,
        ingredient_ids
    ):

        visual_feature = self.visual_encoder(
            image
        )

        emb = self.embedding(
            ingredient_ids
        )

        mask = (
            ingredient_ids != PAD_IDX
        ).unsqueeze(-1)

        emb = emb * mask

        ingredient_feature = (
            emb.sum(dim=1)
            /
            mask.sum(dim=1).clamp(min=1)
        )

        fused = torch.cat(
            [
                visual_feature,
                ingredient_feature
            ],
            dim=1
        )

        return self.fusion(fused)

In [34]:
device = torch.device("cuda:0")

model_vi = VisualIngredientFusion(
    VOCAB_SIZE
).to(device)

images = images.to(device)
ingredients_batch = ingredients_batch.to(device)

with torch.no_grad():
    out = model_vi(
        images,
        ingredients_batch
    )

print("Output:", out.shape)

params = sum(
    p.numel()
    for p in model_vi.parameters()
)

print(
    f"Parameters: {params / 1e6:.2f} M"
)

Output: torch.Size([64, 1])
Parameters: 24.60 M


In [35]:
# ============================================================
# EXP 5A - VISUAL + INGREDIENT FUSION - 1 GPU
# ============================================================

import gc
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# ------------------------------------------------------------
# CLEAN GPU
# ------------------------------------------------------------
try:
    del model_vi
except:
    pass

try:
    del images, ingredients_batch, out
except:
    pass

gc.collect()
torch.cuda.empty_cache()


# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
SEED = 42
EPOCHS = 20
LR = 1e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda:0")


# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------
model_vi = VisualIngredientFusion(
    VOCAB_SIZE
).to(device)

criterion = nn.SmoothL1Loss()

optimizer = torch.optim.Adam(
    model_vi.parameters(),
    lr=LR
)


# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------
def train_one_epoch_vi(
    model,
    loader,
    criterion,
    optimizer,
    device
):
    model.train()

    total_loss = 0.0

    for images, ingredient_ids, calories in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        ingredient_ids = ingredient_ids.to(
            device,
            non_blocking=True
        )

        calories = calories.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad()

        outputs = model(
            images,
            ingredient_ids
        ).squeeze(1)

        loss = criterion(
            outputs,
            calories
        )

        loss.backward()
        optimizer.step()

        total_loss += (
            loss.item() * images.size(0)
        )

    return total_loss / len(loader.dataset)


# ------------------------------------------------------------
# EVALUATION
# ------------------------------------------------------------
def evaluate_vi(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    total_loss = 0.0
    preds = []
    targets = []

    with torch.no_grad():

        for images, ingredient_ids, calories in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            ingredient_ids = ingredient_ids.to(
                device,
                non_blocking=True
            )

            calories = calories.to(
                device,
                non_blocking=True
            )

            outputs = model(
                images,
                ingredient_ids
            ).squeeze(1)

            loss = criterion(
                outputs,
                calories
            )

            total_loss += (
                loss.item() * images.size(0)
            )

            preds.extend(
                outputs.cpu().numpy()
            )

            targets.extend(
                calories.cpu().numpy()
            )

    preds = np.array(preds)
    targets = np.array(targets)

    mae = np.mean(
        np.abs(preds - targets)
    )

    rmse = np.sqrt(
        np.mean(
            (preds - targets) ** 2
        )
    )

    return (
        total_loss / len(loader.dataset),
        mae,
        rmse
    )


# ------------------------------------------------------------
# TRAIN 20 EPOCH
# ------------------------------------------------------------
history_vi = []

best_mae = float("inf")
best_rmse = None
best_epoch = None

total_train_time = 0.0
max_vram_gb = 0.0


for epoch in range(EPOCHS):

    torch.cuda.reset_peak_memory_stats()

    torch.cuda.synchronize()
    start = time.time()

    train_loss = train_one_epoch_vi(
        model_vi,
        train_ing_loader,
        criterion,
        optimizer,
        device
    )

    torch.cuda.synchronize()

    epoch_time = time.time() - start
    total_train_time += epoch_time

    peak_vram = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )

    max_vram_gb = max(
        max_vram_gb,
        peak_vram
    )

    # validation không tính vào training time
    val_loss, val_mae, val_rmse = evaluate_vi(
        model_vi,
        val_ing_loader,
        criterion,
        device
    )

    history_vi.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_mae": val_mae,
        "val_rmse": val_rmse,
        "train_time_sec": epoch_time,
        "peak_vram_gb": peak_vram
    })

    if val_mae < best_mae:

        best_mae = val_mae
        best_rmse = val_rmse
        best_epoch = epoch + 1

        torch.save(
            model_vi.state_dict(),
            "/kaggle/working/"
            "visual_ingredient_20epoch_1gpu_best.pth"
        )

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.2f} | "
        f"MAE: {val_mae:.2f} kcal | "
        f"RMSE: {val_rmse:.2f} kcal | "
        f"Train: {epoch_time:.2f}s | "
        f"VRAM: {peak_vram:.2f} GB"
    )


# ------------------------------------------------------------
# SAVE HISTORY
# ------------------------------------------------------------
pd.DataFrame(
    history_vi
).to_csv(
    "/kaggle/working/"
    "visual_ingredient_20epoch_1gpu_history.csv",
    index=False
)


# ------------------------------------------------------------
# RESULT
# ------------------------------------------------------------
throughput = (
    len(train_ing_dataset)
    * EPOCHS
    / total_train_time
)

print(
    "\n=== VISUAL + INGREDIENT - "
    "1 GPU - 20 EPOCH RESULT ==="
)

print(
    f"Training time: "
    f"{total_train_time:.2f} s"
)

print(
    f"Throughput: "
    f"{throughput:.2f} samples/s"
)

print(
    f"Best epoch: "
    f"{best_epoch}"
)

print(
    f"Best MAE: "
    f"{best_mae:.2f} kcal"
)

print(
    f"Best RMSE: "
    f"{best_rmse:.2f} kcal"
)

print(
    f"Final MAE: "
    f"{history_vi[-1]['val_mae']:.2f} kcal"
)

print(
    f"Final RMSE: "
    f"{history_vi[-1]['val_rmse']:.2f} kcal"
)

print(
    f"Peak training VRAM: "
    f"{max_vram_gb:.2f} GB"
)

Epoch 01/20 | Train Loss: 241.24 | MAE: 124.61 kcal | RMSE: 173.70 kcal | Train: 25.26s | VRAM: 5.49 GB
Epoch 02/20 | Train Loss: 150.76 | MAE: 130.80 kcal | RMSE: 174.65 kcal | Train: 26.86s | VRAM: 5.49 GB
Epoch 03/20 | Train Loss: 78.35 | MAE: 71.56 kcal | RMSE: 102.89 kcal | Train: 28.16s | VRAM: 5.49 GB
Epoch 04/20 | Train Loss: 64.49 | MAE: 65.07 kcal | RMSE: 95.53 kcal | Train: 26.68s | VRAM: 5.49 GB
Epoch 05/20 | Train Loss: 55.68 | MAE: 55.84 kcal | RMSE: 88.05 kcal | Train: 27.66s | VRAM: 5.49 GB
Epoch 06/20 | Train Loss: 50.11 | MAE: 56.71 kcal | RMSE: 89.54 kcal | Train: 27.46s | VRAM: 5.49 GB
Epoch 07/20 | Train Loss: 45.93 | MAE: 57.49 kcal | RMSE: 90.06 kcal | Train: 27.11s | VRAM: 5.49 GB
Epoch 08/20 | Train Loss: 42.15 | MAE: 54.90 kcal | RMSE: 84.27 kcal | Train: 27.66s | VRAM: 5.49 GB
Epoch 09/20 | Train Loss: 47.61 | MAE: 68.05 kcal | RMSE: 106.76 kcal | Train: 27.35s | VRAM: 5.49 GB
Epoch 10/20 | Train Loss: 39.77 | MAE: 55.09 kcal | RMSE: 86.57 kcal | Train: 27.47

In [36]:
# ============================================================
# EXP 5B - STEP 1
# Save fixed splits
# ============================================================

train_ing_df[
    ["dish_id", "calories", "image_path"]
].to_csv(
    "/kaggle/working/train_vi.csv",
    index=False
)

val_ing_df[
    ["dish_id", "calories", "image_path"]
].to_csv(
    "/kaggle/working/val_vi.csv",
    index=False
)

test_ing_df[
    ["dish_id", "calories", "image_path"]
].to_csv(
    "/kaggle/working/test_vi.csv",
    index=False
)

print("Saved:")
print(len(train_ing_df), len(val_ing_df), len(test_ing_df))

Saved:
2270 487 487


In [37]:
%%writefile /kaggle/working/train_vi_ddp.py

import os
import time
import random
import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.distributed as dist

from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler

from torch.nn.parallel import DistributedDataParallel as DDP

from torchvision import transforms
from torchvision.models import (
    resnet50,
    ResNet50_Weights
)


# ============================================================
# CONFIG
# ============================================================

SEED = 42
EPOCHS = 20
LR = 1e-4

LOCAL_BATCH = 32
MAX_INGREDIENTS = 34

PAD_IDX = 0
UNK_IDX = 1

BASE = (
    "/kaggle/input/datasets/"
    "gillesokhin/nutrition5k-dataset"
)

TRAIN_CSV = "/kaggle/working/train_vi.csv"
VAL_CSV = "/kaggle/working/val_vi.csv"

INGR_CSV = f"{BASE}/dish_ingredients.csv"


# ============================================================
# SETUP DDP
# ============================================================

def setup_ddp():

    dist.init_process_group(
        backend="nccl"
    )

    rank = dist.get_rank()
    local_rank = int(
        os.environ["LOCAL_RANK"]
    )

    torch.cuda.set_device(local_rank)

    return rank, local_rank


# ============================================================
# DATA PREPARATION
# ============================================================

def prepare_data():

    train_df = pd.read_csv(TRAIN_CSV)
    val_df = pd.read_csv(VAL_CSV)

    dish_ing = pd.read_csv(
        INGR_CSV
    )[
        ["dish_id", "ingr_id"]
    ].drop_duplicates()

    # --------------------------------
    # vocabulary TRAIN ONLY
    # --------------------------------

    train_dish_ids = set(
        train_df["dish_id"]
    )

    train_ingr_ids = (
        dish_ing[
            dish_ing["dish_id"].isin(
                train_dish_ids
            )
        ]["ingr_id"]
        .unique()
    )

    ingredient_to_idx = {
        ingr_id: idx + 2
        for idx, ingr_id
        in enumerate(
            sorted(train_ingr_ids)
        )
    }

    vocab_size = (
        len(ingredient_to_idx) + 2
    )

    dish_to_ingredients = (
        dish_ing
        .groupby("dish_id")["ingr_id"]
        .apply(list)
        .to_dict()
    )

    def add_indices(df):

        result = df.copy()

        def encode(dish_id):

            ingr_list = (
                dish_to_ingredients
                .get(dish_id, [])
            )

            indices = [
                ingredient_to_idx.get(
                    x,
                    UNK_IDX
                )
                for x in ingr_list
            ]

            indices = (
                indices[
                    :MAX_INGREDIENTS
                ]
            )

            indices += (
                [PAD_IDX]
                * (
                    MAX_INGREDIENTS
                    - len(indices)
                )
            )

            return indices

        result[
            "ingredient_indices"
        ] = result[
            "dish_id"
        ].apply(encode)

        return result

    train_df = add_indices(
        train_df
    )

    val_df = add_indices(
        val_df
    )

    return (
        train_df,
        val_df,
        vocab_size
    )


# ============================================================
# DATASET
# ============================================================

train_transform = transforms.Compose([
    transforms.Resize(
        (224, 224)
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


val_transform = transforms.Compose([
    transforms.Resize(
        (224, 224)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


class NutritionIngredientDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        image = self.transform(
            image
        )

        ingredient_ids = torch.tensor(
            row["ingredient_indices"],
            dtype=torch.long
        )

        calories = torch.tensor(
            row["calories"],
            dtype=torch.float32
        )

        return (
            image,
            ingredient_ids,
            calories
        )


# ============================================================
# MODEL
# ============================================================

class VisualIngredientFusion(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        embedding_dim=64
    ):

        super().__init__()

        self.visual_encoder = resnet50(
            weights=ResNet50_Weights.DEFAULT
        )

        visual_dim = (
            self.visual_encoder
            .fc
            .in_features
        )

        self.visual_encoder.fc = (
            nn.Identity()
        )

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_IDX
        )

        self.fusion = nn.Sequential(
            nn.Linear(
                visual_dim
                + embedding_dim,
                512
            ),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(
                512,
                1
            )
        )

    def forward(
        self,
        image,
        ingredient_ids
    ):

        visual_feature = (
            self.visual_encoder(
                image
            )
        )

        emb = self.embedding(
            ingredient_ids
        )

        mask = (
            ingredient_ids
            != PAD_IDX
        ).unsqueeze(-1)

        emb = emb * mask

        ingredient_feature = (
            emb.sum(dim=1)
            /
            mask.sum(
                dim=1
            ).clamp(min=1)
        )

        fused = torch.cat(
            [
                visual_feature,
                ingredient_feature
            ],
            dim=1
        )

        return self.fusion(
            fused
        )


# ============================================================
# TRAIN
# ============================================================

def train_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    loss_sum = 0.0
    sample_count = 0

    for (
        images,
        ingredient_ids,
        calories
    ) in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        ingredient_ids = (
            ingredient_ids.to(
                device,
                non_blocking=True
            )
        )

        calories = calories.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad()

        outputs = model(
            images,
            ingredient_ids
        ).squeeze(1)

        loss = criterion(
            outputs,
            calories
        )

        loss.backward()
        optimizer.step()

        loss_sum += (
            loss.item()
            * images.size(0)
        )

        sample_count += (
            images.size(0)
        )

    stats = torch.tensor(
        [
            loss_sum,
            sample_count
        ],
        dtype=torch.float64,
        device=device
    )

    dist.all_reduce(
        stats,
        op=dist.ReduceOp.SUM
    )

    return (
        stats[0].item()
        / stats[1].item()
    )


# ============================================================
# VALIDATION
# IMPORTANT:
# use model.module on rank 0 only
# ============================================================

def evaluate(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0.0
    preds = []
    targets = []

    with torch.no_grad():

        for (
            images,
            ingredient_ids,
            calories
        ) in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            ingredient_ids = (
                ingredient_ids.to(
                    device,
                    non_blocking=True
                )
            )

            calories = calories.to(
                device,
                non_blocking=True
            )

            outputs = model(
                images,
                ingredient_ids
            ).squeeze(1)

            loss = criterion(
                outputs,
                calories
            )

            total_loss += (
                loss.item()
                * images.size(0)
            )

            preds.extend(
                outputs.cpu().numpy()
            )

            targets.extend(
                calories.cpu().numpy()
            )

    preds = np.array(preds)
    targets = np.array(targets)

    mae = np.mean(
        np.abs(
            preds - targets
        )
    )

    rmse = np.sqrt(
        np.mean(
            (preds - targets) ** 2
        )
    )

    return (
        total_loss
        / len(loader.dataset),
        mae,
        rmse
    )


# ============================================================
# MAIN
# ============================================================

def main():

    rank, local_rank = setup_ddp()

    device = torch.device(
        f"cuda:{local_rank}"
    )

    # SAME MODEL SEED
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    (
        train_df,
        val_df,
        vocab_size
    ) = prepare_data()

    train_dataset = (
        NutritionIngredientDataset(
            train_df,
            train_transform
        )
    )

    val_dataset = (
        NutritionIngredientDataset(
            val_df,
            val_transform
        )
    )

    train_sampler = (
        DistributedSampler(
            train_dataset,
            shuffle=True,
            seed=SEED
        )
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=LOCAL_BATCH,
        sampler=train_sampler,
        num_workers=2,
        pin_memory=True
    )

    # rank 0 evaluates full validation
    val_loader = None

    if rank == 0:

        val_loader = DataLoader(
            val_dataset,
            batch_size=64,
            shuffle=False,
            num_workers=2,
            pin_memory=True
        )

    model = (
        VisualIngredientFusion(
            vocab_size
        ).to(device)
    )

    model = DDP(
        model,
        device_ids=[local_rank]
    )

    criterion = (
        nn.SmoothL1Loss()
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR
    )

    history = []

    best_mae = float("inf")
    best_rmse = None
    best_epoch = None

    total_train_time = 0.0
    max_vram = 0.0

    for epoch in range(EPOCHS):

        train_sampler.set_epoch(
            epoch
        )

        torch.cuda.reset_peak_memory_stats(
            device
        )

        # --------------------------------
        # synchronized training timer
        # --------------------------------

        dist.barrier()
        torch.cuda.synchronize()

        start = time.time()

        train_loss = train_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device
        )

        torch.cuda.synchronize()
        dist.barrier()

        epoch_time = (
            time.time() - start
        )

        # max time across GPUs
        time_tensor = torch.tensor(
            epoch_time,
            device=device
        )

        dist.all_reduce(
            time_tensor,
            op=dist.ReduceOp.MAX
        )

        epoch_time = (
            time_tensor.item()
        )

        # --------------------------------
        # VRAM
        # --------------------------------

        local_vram = (
            torch.cuda
            .max_memory_allocated(
                device
            )
            / 1024**3
        )

        vram_tensor = torch.tensor(
            local_vram,
            device=device
        )

        dist.all_reduce(
            vram_tensor,
            op=dist.ReduceOp.MAX
        )

        epoch_vram = (
            vram_tensor.item()
        )

        if rank == 0:

            total_train_time += (
                epoch_time
            )

            max_vram = max(
                max_vram,
                epoch_vram
            )

            # IMPORTANT:
            # don't use DDP wrapper for
            # rank-0-only validation
            val_loss, val_mae, val_rmse = (
                evaluate(
                    model.module,
                    val_loader,
                    criterion,
                    device
                )
            )

            history.append({
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_mae": val_mae,
                "val_rmse": val_rmse,
                "train_time_sec": epoch_time,
                "peak_vram_gb": epoch_vram
            })

            if val_mae < best_mae:

                best_mae = val_mae
                best_rmse = val_rmse
                best_epoch = epoch + 1

                torch.save(
                    model.module.state_dict(),
                    "/kaggle/working/"
                    "visual_ingredient_"
                    "20epoch_2gpu_ddp_best.pth"
                )

            print(
                f"Epoch "
                f"{epoch+1:02d}/{EPOCHS} | "
                f"Train Loss: "
                f"{train_loss:.2f} | "
                f"MAE: "
                f"{val_mae:.2f} kcal | "
                f"RMSE: "
                f"{val_rmse:.2f} kcal | "
                f"Train: "
                f"{epoch_time:.2f}s | "
                f"VRAM/GPU: "
                f"{epoch_vram:.2f} GB"
            )

        # rank 1 waits for validation
        dist.barrier()

    # ========================================================
    # FINAL RESULT
    # ========================================================

    if rank == 0:

        history_df = pd.DataFrame(
            history
        )

        history_df.to_csv(
            "/kaggle/working/"
            "visual_ingredient_"
            "20epoch_2gpu_history.csv",
            index=False
        )

        throughput = (
            len(train_dataset)
            * EPOCHS
            / total_train_time
        )

        print(
            "\n=== VISUAL + INGREDIENT - "
            "2 GPU DDP - 20 EPOCH RESULT ==="
        )

        print(
            f"Training time: "
            f"{total_train_time:.2f} s"
        )

        print(
            f"Throughput: "
            f"{throughput:.2f} samples/s"
        )

        print(
            f"Best epoch: "
            f"{best_epoch}"
        )

        print(
            f"Best MAE: "
            f"{best_mae:.2f} kcal"
        )

        print(
            f"Best RMSE: "
            f"{best_rmse:.2f} kcal"
        )

        print(
            f"Final MAE: "
            f"{history[-1]['val_mae']:.2f} kcal"
        )

        print(
            f"Final RMSE: "
            f"{history[-1]['val_rmse']:.2f} kcal"
        )

        print(
            f"Peak VRAM per GPU: "
            f"{max_vram:.2f} GB"
        )

    dist.destroy_process_group()


if __name__ == "__main__":
    main()

Writing /kaggle/working/train_vi_ddp.py


In [38]:
import torch

print(
    "GPU count:",
    torch.cuda.device_count()
)

for i in range(
    torch.cuda.device_count()
):
    print(
        i,
        torch.cuda.get_device_name(i)
    )

GPU count: 2
0 Tesla T4
1 Tesla T4


In [39]:
!torchrun \
    --standalone \
    --nproc_per_node=2 \
    /kaggle/working/train_vi_ddp.py

W0912 03:55:06.477000 791 torch/distributed/run.py:852] 
W0912 03:55:06.477000 791 torch/distributed/run.py:852] *****************************************
W0912 03:55:06.477000 791 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0912 03:55:06.477000 791 torch/distributed/run.py:852] *****************************************
[W912 03:55:06.465494598 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W912 03:55:10.782369561 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W912 03:55:10.826495837 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can sp

In [41]:
test_ing_loader = DataLoader(
    test_ing_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [42]:
# ============================================================
# EXP 5 - TEST FUNCTION
# ============================================================

import numpy as np
import torch

def test_vi_model(
    model,
    loader,
    device
):
    model.eval()

    preds = []
    targets = []

    with torch.no_grad():

        for images, ingredient_ids, calories in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            ingredient_ids = ingredient_ids.to(
                device,
                non_blocking=True
            )

            outputs = model(
                images,
                ingredient_ids
            ).squeeze(1)

            preds.extend(
                outputs.cpu().numpy()
            )

            targets.extend(
                calories.numpy()
            )

    preds = np.array(preds)
    targets = np.array(targets)

    mae = np.mean(
        np.abs(preds - targets)
    )

    rmse = np.sqrt(
        np.mean(
            (preds - targets) ** 2
        )
    )

    return mae, rmse, preds, targets

In [43]:
# ============================================================
# TEST - EXP 5A - 1 GPU BEST CHECKPOINT
# ============================================================

device = torch.device("cuda:0")

model_vi_1gpu = VisualIngredientFusion(
    VOCAB_SIZE
).to(device)

model_vi_1gpu.load_state_dict(
    torch.load(
        "/kaggle/working/"
        "visual_ingredient_20epoch_1gpu_best.pth",
        map_location=device
    )
)

mae_1gpu, rmse_1gpu, preds_1gpu, targets = (
    test_vi_model(
        model_vi_1gpu,
        test_ing_loader,
        device
    )
)

print(
    "=== VISUAL + INGREDIENT - 1 GPU TEST ==="
)

print(
    f"MAE : {mae_1gpu:.2f} kcal"
)

print(
    f"RMSE: {rmse_1gpu:.2f} kcal"
)

=== VISUAL + INGREDIENT - 1 GPU TEST ===
MAE : 49.85 kcal
RMSE: 83.89 kcal


In [45]:
import os

p1 = "/kaggle/working/visual_ingredient_20epoch_1gpu_best.pth"
p2 = "/kaggle/working/visual_ingredient_20epoch_2gpu_ddp_best.pth"

print("1 GPU:", os.path.exists(p1))
print("2 GPU:", os.path.exists(p2))

1 GPU: True
2 GPU: True


In [46]:
import numpy as np
import torch

def test_vi_model(model, loader, device):

    model.eval()

    preds = []
    targets = []

    with torch.no_grad():

        for images, ingredient_ids, calories in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            ingredient_ids = ingredient_ids.to(
                device,
                non_blocking=True
            )

            outputs = model(
                images,
                ingredient_ids
            ).squeeze(1)

            preds.extend(
                outputs.cpu().numpy()
            )

            targets.extend(
                calories.numpy()
            )

    preds = np.array(preds)
    targets = np.array(targets)

    mae = np.mean(
        np.abs(preds - targets)
    )

    rmse = np.sqrt(
        np.mean(
            (preds - targets) ** 2
        )
    )

    return mae, rmse

In [47]:
device = torch.device("cuda:0")

model_vi_1gpu = VisualIngredientFusion(
    VOCAB_SIZE
).to(device)

model_vi_1gpu.load_state_dict(
    torch.load(
        "/kaggle/working/"
        "visual_ingredient_20epoch_1gpu_best.pth",
        map_location=device
    )
)

mae_1gpu, rmse_1gpu = test_vi_model(
    model_vi_1gpu,
    test_ing_loader,
    device
)

print("=== 1 GPU TEST ===")
print(f"MAE : {mae_1gpu:.2f} kcal")
print(f"RMSE: {rmse_1gpu:.2f} kcal")

=== 1 GPU TEST ===
MAE : 49.85 kcal
RMSE: 83.89 kcal


In [48]:
model_vi_2gpu = VisualIngredientFusion(
    VOCAB_SIZE
).to(device)

model_vi_2gpu.load_state_dict(
    torch.load(
        "/kaggle/working/"
        "visual_ingredient_20epoch_2gpu_ddp_best.pth",
        map_location=device
    )
)

mae_2gpu, rmse_2gpu = test_vi_model(
    model_vi_2gpu,
    test_ing_loader,
    device
)

print("=== 2 GPU DDP TEST ===")
print(f"MAE : {mae_2gpu:.2f} kcal")
print(f"RMSE: {rmse_2gpu:.2f} kcal")

=== 2 GPU DDP TEST ===
MAE : 52.30 kcal
RMSE: 85.04 kcal
